In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

speciesdata = pd.read_csv(our_data + "species_ligand.txt", sep="\t", header=0)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
deletedata = speciesdata[speciesdata["species"] == "Arabidopsis thaliana"]
deletedata = speciesdata[speciesdata["species"] == "Arabidopsis thaliana"]
deletedata = speciesdata[speciesdata["species"] == "Arabidopsis thaliana"]
deletedata = speciesdata[speciesdata["species"] == "Arabidopsis thaliana"]

In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)

p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [ ]:
filtered_rows = p450plant[
    (p450plant["Binding"] == 1) & p450plant["enzyme"].isin(deletedata["P450 name"])
]
p450plant_filtered = p450plant.drop(filtered_rows.index)
p450plant_filtered.reset_index(drop=True, inplace=True)

In [ ]:
df_train = pd.concat([p450plant_filtered, df_train], ignore_index=True)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


train_X, train_y = create_input_and_output_data(df=df_train)
test_X, test_y = create_input_and_output_data(df=df_test)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
param = {
    "learning_rate": 0.31553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}


num_round = param["num_rounds"]
param["objective"] = "binary:logistic"

weights = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in np.array(train_y)]
)

del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)

evallist = [(dtrain, "train")]
bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))

[0]	train-error:0.28472
[10]	train-error:0.13516
[20]	train-error:0.08098
[30]	train-error:0.05767
[40]	train-error:0.04362
[50]	train-error:0.03474
[60]	train-error:0.02717
[70]	train-error:0.02211
[80]	train-error:0.01713
[90]	train-error:0.01448
[100]	train-error:0.01229
[110]	train-error:0.01047
[120]	train-error:0.00908
[130]	train-error:0.00753
[140]	train-error:0.00660
[150]	train-error:0.00563
[160]	train-error:0.00485
[170]	train-error:0.00452
[180]	train-error:0.00410
[190]	train-error:0.00355
[200]	train-error:0.00311
[210]	train-error:0.00291
[220]	train-error:0.00269
[230]	train-error:0.00250
[240]	train-error:0.00227
[250]	train-error:0.00211
[260]	train-error:0.00199
[270]	train-error:0.00196
[280]	train-error:0.00181
[290]	train-error:0.00167
[300]	train-error:0.00159
[310]	train-error:0.00154
[320]	train-error:0.00141
[330]	train-error:0.00137
[340]	train-error:0.00132
[341]	train-error:0.00132
Accuracy on test set: 0.8948311921123394, ROC-AUC score for test set: 0.933

In [ ]:
pickle.dump(bst, open(join(our_data + "deletedatamodel.dat"), "wb"))

In [10]:
p450plant_filtered

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP716E41,MAS,1,0000000000000000000000000000000001001000000000...,"[-0.1127784475684166, 0.23461320996284485, 0.0..."
1,CYP726A17,CAS,1,0000000000000000000000000100000001001000000000...,"[-0.07275320589542389, 0.12327290326356888, -0..."
2,CYP71D15,LIM,1,0000000000000000000000000000000001000000000000...,"[-0.0860673263669014, 0.16397367417812347, 0.0..."
3,CYP72A66v2,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.0892251580953598, 0.15670408308506012, -0...."
4,CYP76AH15,MOX,1,0000100000000000000000000000000001001000000000...,"[-0.04311390966176987, 0.1504163295030594, -0...."
...,...,...,...,...,...
1193,CYP90G4,FRA,0,0000000000000000000000000000000001000000000000...,"[-0.1249723955988884, 0.2523147463798523, 0.07..."
1194,CYP72A610,TSP,0,0000000000000000000000000100000100000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1195,CYP72A610,PUL,0,0000000000000000000000000000000001000000000000...,"[-0.07047156989574432, 0.16309861838817596, 0...."
1196,CYP79D1,JAS,0,0000000000000000000000000000000001000000000000...,"[-0.01571257971227169, 0.19178493320941925, 0...."


In [11]:
deletedata

,P450 name,docking score,ranking,p450 num,substrate,species
0,CYP706A3,0.589,9,230,ABA,Arabidopsis thaliana
1,CYP716A2,0.657,94,230,BAM,Arabidopsis thaliana
2,CYP710A1,0.614,22,230,BSI,Arabidopsis thaliana
3,CYP85A2,0.827,2,230,CAT,Arabidopsis thaliana
4,CYP76C2,0.810,1,230,CIT,Arabidopsis thaliana
5,CYP76C4,0.731,5,230,CIT,Arabidopsis thaliana
6,CYP98A3,0.425,81,230,COU,Arabidopsis thaliana
7,CYP82G1,0.706,3,230,EGE,Arabidopsis thaliana
8,CYP84A1,0.643,7,230,FEU,Arabidopsis thaliana
9,CYP714A2,0.435,82,230,GIA,Arabidopsis thaliana
